In [125]:
import numpy as np
import pandas as pd
from stabl import data
from stabl.multi_omic_pipelines import multi_omic_stabl_cv
from sklearn.model_selection import RepeatedStratifiedKFold, GroupShuffleSplit, GridSearchCV
from sklearn.linear_model import LogisticRegression
from stabl.stabl import Stabl
from stabl.adaptive import ALogitLasso
from sklearn.base import clone

import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from stabl.stabl import Stabl, plot_stabl_path, plot_fdr_graph, export_stabl_to_csv, save_stabl_results
from stabl.preprocessing import LowInfoFilter
from stabl.visualization import boxplot_features, scatterplot_features, plot_roc, boxplot_binary_predictions
from stabl.adaptive import ALasso, ALogitLasso
from stabl import data

from sklearn.pipeline import Pipeline
from sklearn.feature_selection import VarianceThreshold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.base import clone
from sklearn.linear_model import Lasso, LogisticRegression, ElasticNet

import pandas as pd
import numpy as np
#from .preprocessing import remove_low_info_samples
from os.path import join
import os
import stabl.multi_omic_pipelines
print(stabl.multi_omic_pipelines)
import stabl
print(stabl.__file__)
import sys
print(sys.executable)

%config InlineBackend.figure_formats=['retina'] 


# Defining outer Cross-Validation (CV) loop and inner CV loop
outter_cv = GroupShuffleSplit(n_splits=100, test_size=0.2, random_state=42)
chosen_inner_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

# Type of artificial features to generate.
artificial_type = "knockoff" #"random_permutation"  # or "knockoff"

# Lasso definition
lasso = LogisticRegression(
    penalty="l1", class_weight="balanced", max_iter=int(1e6), solver="liblinear", random_state=42
)
lasso_cv = GridSearchCV(
    lasso, param_grid={"C": np.logspace(-2, 2, 30)}, scoring="roc_auc", cv=chosen_inner_cv, n_jobs=-1
)  # This is the lasso in CV, tuned at each fold

# ElasticNet definition
en = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    class_weight='balanced',
    max_iter=int(1e3),
    random_state=42
)
en_cv = GridSearchCV(
    en, param_grid={"C": np.logspace(-2, 1, 5), "l1_ratio": [.5, .7, .9]},
    scoring="roc_auc", cv=chosen_inner_cv, n_jobs=-1
)

# Adaptive Lasso definition
alasso = ALogitLasso(
    penalty="l1", solver="saga", max_iter=int(1e6), class_weight='balanced', random_state=42
) #liblinear
alasso_cv = GridSearchCV(
    alasso, scoring='roc_auc', param_grid={"C": np.logspace(-2, 2, 20)}, cv=chosen_inner_cv, n_jobs=-1
)

# Stabl definition
stabl = Stabl(
    base_estimator=lasso,
    n_bootstraps=150, 
    artificial_type=artificial_type,
    artificial_proportion=.5,
    replace=False,
    fdr_threshold_range=np.arange(0.1, 1, 0.01),
    sample_fraction=0.5,
    random_state=42,
    lambda_grid={"C": np.linspace(0.01, 1, 10)},
    verbose=1
)  # Base Stabl definition with Lasso SRM

stabl_alasso = clone(stabl).set_params(
    base_estimator=alasso,
    lambda_grid={"C": np.linspace(0.01, 10, 10)},
    verbose=1
)  # Stabl_ALasso

stabl_en = clone(stabl).set_params(
    base_estimator=en,
    n_bootstraps=50,
    lambda_grid=[
        {"C": np.logspace(-3, -1, 5), "l1_ratio": [0.9]}
    ],
    verbose=1
)  # Stabl_EN

# Overall frameworks, don't comment them out, they must be declared even when not all are used
estimators = {
    "lasso": lasso_cv,
    "alasso": alasso_cv,
    "en": en_cv,
    "stabl_lasso": stabl,
    "stabl_alasso": stabl_alasso,
    "stabl_en": stabl_en
}

# Associated model names. Here you can comment what you don't want to test
models = [
    "STABL Lasso",
    "Lasso",
    "STABL ALasso",
    "ALasso",
    # "STABL ElasticNet",
    # "ElasticNet"
]

# For All Omics With Lymph

In [143]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_with_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
        "Cell": cell_train,
  "SNV": snv_train,
    "CNA": cna_train,
   "Menv": menv_train,
    "Marker": marker_train,
    "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_All_with_Lymph_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)

# For All Omics without Lymph

In [ ]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_no_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
        "Cell": cell_train,
  "SNV": snv_train,
    "CNA": cna_train,
   "Menv": menv_train,
    "Marker": marker_train,
    "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_All_without_Lymph_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)

# For IMC

In [ ]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_no_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
       "Cell": cell_train,
 # "SNV": snv_train,
 #   "CNA": cna_train,
   "Menv": menv_train,
    "Marker": marker_train,
  #  "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_IMC_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)

# For Genomics

In [1]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_no_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
  #     "Cell": cell_train,
  "SNV": snv_train,
    "CNA": cna_train,
 #  "Menv": menv_train,
  #  "Marker": marker_train,
  #  "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_Genomics_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)

# For Clinical with Lymph

In [ ]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_with_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
  #     "Cell": cell_train,
#  "SNV": snv_train,
 #   "CNA": cna_train,
 #  "Menv": menv_train,
  #  "Marker": marker_train,
    "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_Clinical_with_Lymph_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)

# For Clinical without Lymph

In [ ]:
data_path = ""
#Load all data
snv_train = pd.read_csv( 
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_mutations_in_10plus_patients.csv"), index_col="PCSI_ID") 
cell_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_celltype_densities.csv"), index_col="PCSI_ID") 
menv_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_microenvironment_densities.csv"), index_col="PCSI_ID")
cna_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "GENOMIC_copy_number_and_rearrangement_counts.csv"), index_col="PCSI_ID") 
marker_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "IMC_marker_intensities.csv"), index_col="PCSI_ID")
clinical_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "CLINICAL_no_lymph.csv"), index_col="PCSI_ID")
early_fusion = pd.read_csv(
        join(data_path, "EarlyFusion_Crossomics.csv"), index_col="PCSI_ID") 
X_train = {
  #     "Cell": cell_train,
#  "SNV": snv_train,
 #   "CNA": cna_train,
 #  "Menv": menv_train,
  #  "Marker": marker_train,
    "Clinical": clinical_train,
   # "Early_Fuse": early_fusion
    }

task_type = "binary"

# Load n = 174 binarized overall survival per PCSI_ID
y_train = pd.read_csv(
        join(data_path, "Stabl_Data_No_Neoadjuvant", "Overall_Survival.csv"), index_col="PCSI_ID")
y_train = y_train.astype(int)
common_indices = set(y_train.index).intersection(*(set(df.index) for df in X_train.values()))
common_indices = sorted(common_indices)  # optional: sort for consistent order


# Step 2: Subset all omic DataFrames to these common indices
X_train = {omic: df.loc[common_indices] for omic, df in X_train.items()}

# Step 3: Subset y_train to these common indices
y_train = y_train.loc[common_indices]

ids = pd.Series(index=common_indices, data=range(len(common_indices)))

y_train = y_train["os_cf_binary"].astype(int) 

# Pipeline run in cross-validation. 
print("Run CV on overall survival PDAC dataset")
pred_dict= multi_omic_stabl_cv( 
    data_dict= X_train,
    y= y_train,
    outer_splitter= outter_cv,
    estimators= estimators,
   task_type= task_type, 
    save_path= "Stabl_Clinical_without_Lymph_output",  
    outer_groups= ids, 
    early_fusion= False,
    late_fusion= True,
    n_iter_lf= 1000,
    models= models)